# ConjunctNet Masking

In [ ]:
!pip install ultralytics==8.0.196
!pip install roboflow
!pip install albumentations==1.4

In [ ]:
import os
import ultralytics
ultralytics.checks()
from ultralytics import YOLO

from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# this is for importing dataset images from RoboFlow
# other alternatives may be used
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_HERE")
project = rf.workspace("WORKSPACE").project("PROJECT")
version = project.version(PROJECT_VERSION)
dataset = version.download("yolov8")

In [ ]:
!yolo task=segment mode=train model=yolov8n-seg.pt data="{dataset.location}/data.yaml" epochs=50 imgsz=640 plots=True device=0 name="conjunct_seg" freeze=9

In [ ]:
model = YOLO("yolov8n-seg.pt")
results = model.train(
    data="{dataset.location}/data.yaml", 
    epochs=50, 
    imgsz=640, 
    freeze=9, 
    name="conjunct_seg",
    

In [ ]:
def process_mask(result, dilate=False):
    if result.masks is None:
        return None
    processed_masks = []
    imgh, imgw = result.masks.orig_shape
    for mask in result.masks.data:
        mask = mask.cpu().numpy()
        mask = (mask * 255).astype(np.uint8)
        large_dim = max((imgw, imgh))
        small_dim = min((imgw, imgh))
        pad_one_side = (large_dim - small_dim) // 2
        mask = cv2.resize(mask, (large_dim, large_dim))
        mask = mask[pad_one_side:pad_one_side+imgh, :imgw]
        processed_masks.append(mask)
    mask = np.zeros_like(processed_masks[0])
    for m in processed_masks:
        mask = cv2.bitwise_or(mask, m)
    if dilate:
      kernel = np.ones((3,3),np.uint8)
      mask = cv2.dilate(mask, kernel, iterations=1, borderType=cv2.BORDER_CONSTANT, borderValue=0)
    return mask

In [ ]:
# Load the trained model weights
model = YOLO(model_path)

results = []
images_to_process = [LIST_OF_IMAGE_PATHS]
output_dir = "/path/to/output/directory/"
images = []

with torch.no_grad():
  for img in images_to_process:
      images.append(img)
      img_count += 1
    results = model.predict(images, save=False, imgsz=640, conf=0.9, device=device)
    results_mask = [process_mask(result, dilate=False) for result in results]
    img_names = [img.split("/")[-1] for img in images]
    for i, result in enumerate(results_mask):
        # if result is None, skip and discard the image
        if result is None:
        continue
        img_name = img_names[i]
        cv2.imwrite(f'{output_dir}/{img_name}', result)
        del result
        torch.cuda.empty_cache()
        gc.collect()
        images = []
print(len(results))